# Case Study 6.1 - Classification with LLMs

In the first case study of chapter 6 we look at using a generative language model API to perform what is known as zero shot classification. This means we ask the model to classify data for us using just a prompted description of the task.

## 01 - Data Preparation

We will demonstrate the zero to few shot technique using a data set of website comments.

In [ ]:
from datasets import load_dataset
from datasets import get_dataset_split_names

In [ ]:
get_dataset_split_names("google/civil_comments")

In [ ]:
ds = load_dataset("google/civil_comments", split="train")

In [ ]:
cols = ['toxicity', 'severe_toxicity', 'obscene', 'threat', 'insult', 'identity_attack', 'sexual_explicit']

In [ ]:
# Convert columns to binary representations
def make_binary(sample):
    for x in cols:
        if sample[x] > 0:
            sample[x] = 1.0
    return sample

bin_ds = ds.map(make_binary)

In [ ]:
## Print out percentages of comment types
for x in cols:
    print( str(round(100*np.mean(bin_ds[x]), 1)) + " % " + x)

In [ ]:
# The union of toxic comment types as a proxy for 
def add_proxy(sample):
    scores = [sample[x] for x in cols]
    score = np.sum(scores)
    if score > 0:
        sample['proxy'] = 1.0
    else:
        sample['proxy'] = 0.0
    return sample

updated_ds = ds.map(add_proxy)

In [ ]:
from datasets import concatenate_datasets

shuff = updated_ds.shuffle(seed=42)

samples = 100
pos = shuff.filter(lambda x: x["proxy"]==1).select(range(samples))
neg = shuff.filter(lambda x: x["proxy"]==0).select(range(samples))

case_ds = concatenate_datasets([pos,neg])

## 02 - Create Few Shot Pipeline 

In [ ]:
from pydantic import BaseModel, Field
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import PromptTemplate

In [ ]:
class Sentiment(BaseModel):
    sentiment: str = Field(
        description="Categorisation of the sentiment of text as one of these three values: positive, neutral, negative"
    )

par = JsonOutputParser(pydantic_object=Sentiment)

In [ ]:
prompt = PromptTemplate(
   template = """
     Your job is to classify text from a website comment section based on
     the sentiment expressed by the author. The allowed responses are:
      'positive' - for text expressing positive emotions.
      'negative' - for text expressing negative emotions.
      'neutral' -  for text that expresses no particular emotional content.
     Here are some examples:
      ---
      Text: 'Ohh man, I cannot get enough of this!'
      Response: 'positive'
      ---
      Text: 'This article sucks, the author is stupid.'
      Response: 'negative'
      ---
      Text: 'Shut up and take my money!'
      Response: 'positive'
      ---
      Text: 'Where did they find this guy?'
      Response: 'negative'
      ---
      Text: 'Can someone tell me what the second part means?'
      Response: 'neutral'
      ---
     Here is the text for you to classify:
     {input_text}.
     {format_instructions}
     """,
   input_variables=["input_text"],
   partial_variables={"format_instructions":par.get_format_instructions()}
)


In [ ]:
# Test the prompt to see the substitution

input = "some text to classify "
print(prompt.invoke({"input_text":input}))

## Pipeline

Put the templated prompt into a pipeline with a model and the parser

In [ ]:
from langchain_openai import ChatOpenAI
model = ChatOpenAI(model="gpt-4o", temperature=0)

chain = prompt | model | pars

In [ ]:
# Run some simple tests

response1 = chain.invoke({"input_text":"Yeah, well, this was a pretty dull read."})
response2 = chain.invoke({"input_text":"OMG, where did she get all these ideas?"})

print("Response 1:", response1['sentiment'], "\nResponse 2:", response2['sentiment'])

## Test data

We now set up the structures for running a larger test over the dataset downloaded from HuggingFace at the top of the Notebook.

**Note:** One of the models used in the book was deprecated after the chapter was written. I have commented the affected code sections out and replaced that model with a newer one.

In [ ]:
import pandas as pd

allowed_values = ["positive", "neutral", "negative"]
def test_model(model, prompt, parser, in_ds, in_col, comp_col, out_col ):
   result = pd.DataFrame(columns=[in_col, comp_col, out_col])
   chain = prompt | model | parser
   for x in range(len(in_ds)):
      text = in_ds[x][in_col]
      comp = in_ds[x][comp_col]
      output = ""
      try:
          response = chain.invoke({"input_text":text})
          if out_col in response:
              output = response[out_col]
              if output not in allowed_values:
                  output = "invalid"
          else:
              output = "missing"
      except:
          output = "error"
      record = {in_col:text, comp_col:comp, out_col:output}
      result = pd.concat([result,pd.DataFrame([record])], axis=0, ignore_index=True)
   return result

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_openai import ChatOpenAI

oai = ChatOpenAI(model="gpt-4o", temperature=0)
#gem = ChatGoogleGenerativeAI(model="gemini-1.5-pro")
gem2 = ChatGoogleGenerativeAI(model="gemini-2.0-flash")

models = {
  #"gemini-1.5-pro": gem,
  "gemini-2.0-flash":gen2,
  "gpt-4o": oai
}

In [ ]:
test_results = pd.DataFrame()

for m in models.keys():
    mod = models[m]
    rez = test_model(
        mod, prompt, par, case_ds,
        "text", "proxy", "sentiment"
    )
    rez['model'] = m
    test_results = pd.concat(
        [test_results, rez], axis=0,
        ignore_index=True
    )

In [ ]:
test_results.groupby(['model','proxy','sentiment'])['text'].count()

In [ ]:
temp = test_results[ (test_results ['sentiment']=='positive') &
                     (test_results ['proxy']==1.0)
]

uniqs = temp['text'].unique()
print("Records", len(uniqs), "\n---")
print("\n---\n".join(uniqs))